## 01_IMPORTS

In [6]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import csv
import sys

from IPython.display import display

## 02_PROJECT_PATHS

In [4]:
PROJECT_ROOT = Path(
    r"C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor"
)

DATA_DIR = PROJECT_ROOT / "Data"

RAW_DIR = DATA_DIR / "Raw"
CLEAN_DIR = DATA_DIR / "Clean"
PROCESSED_DIR = DATA_DIR / "Processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATASET_A_PATH = (
    CLEAN_DIR
    / "cleaned_movies.csv"
)

DATASET_B_PATH = (
    RAW_DIR
    / "archive"
    / "movies.csv"
)

DATASET_C_PATH = (
    RAW_DIR
    / "21920642"
    / "movies_raw.csv"
)

print("Project root:", PROJECT_ROOT)
print("Processed output:", PROCESSED_DIR)

Project root: C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor
Processed output: C:\Users\bongo\OneDrive\Desktop\GitHub\Movie Analysis Predictor\Data\Processed


## 03_LOAD_SOURCE_DATASETS

In [7]:
csv.field_size_limit(sys.maxsize)

cleaned_movies = pd.read_csv(
    DATASET_A_PATH,
    encoding="utf-8"
)

tmdb_relational = pd.read_csv(
    DATASET_B_PATH,
    engine="python",
    encoding="utf-8"
)

tmdb_recent = pd.read_csv(
    DATASET_C_PATH,
    encoding="utf-8"
)

print("=" * 80)
print("SOURCE DATASETS LOADED")
print("=" * 80)

print(
    "Dataset A:",
    cleaned_movies.shape
)

print(
    "Dataset B:",
    tmdb_relational.shape
)

print(
    "Dataset C:",
    tmdb_recent.shape
)

SOURCE DATASETS LOADED
Dataset A: (7668, 20)
Dataset B: (9771, 22)
Dataset C: (17978, 16)


## 04_INSPECTING_SOURCE_COLUMNS

In [8]:
print("=" * 80)
print("DATASET A - CLEANED MOVIES")
print("=" * 80)

for column in cleaned_movies.columns:
    print(column)


print("\n" + "=" * 80)
print("DATASET B - TMDB RELATIONAL")
print("=" * 80)

for column in tmdb_relational.columns:
    print(column)


print("\n" + "=" * 80)
print("DATASET C - TMDB RECENT")
print("=" * 80)

for column in tmdb_recent.columns:
    print(column)

DATASET A - CLEANED MOVIES
name
rating
genre
year
released
score
votes
director
writer
star
country
budget
gross
company
runtime
Profit
ROI
released_month
release_quarter
decade

DATASET B - TMDB RELATIONAL
id
title
original_title
overview
release_date
runtime
budget
revenue
vote_average
vote_count
popularity
poster_path
backdrop_path
status
tagline
homepage
original_language
adult
video
created_at
updated_at
genres

DATASET C - TMDB RECENT
movie_id
title
release_date
status
budget
revenue
runtime
genres
production_companies
production_countries
spoken_languages
original_language
vote_average
vote_count
popularity
adult


## DEFINE SOURCE ROLES AND PRIORITY

In [9]:
source_priority = {
    "financial": [
        "A",
        "B",
        "C"
    ],

    "modern_metadata": [
        "C",
        "B",
        "A"
    ],

    "tmdb_metadata": [
        "C",
        "B"
    ],

    "historical_metadata": [
        "A",
        "B",
        "C"
    ]
}


source_roles = {
    "A": {
        "name": "Cleaned Industry Dataset",
        "role": "Historical financial backbone",
        "strengths": [
            "budget",
            "gross revenue",
            "historical coverage",
            "clean financial observations"
        ]
    },

    "B": {
        "name": "TMDB Relational Dataset",
        "role": "Relational metadata and enrichment",
        "strengths": [
            "TMDB metadata",
            "genre relationships",
            "cast and crew relationships",
            "production metadata"
        ]
    },

    "C": {
        "name": "TMDB Recent Dataset",
        "role": "Modern movie coverage and enrichment",
        "strengths": [
            "modern movie coverage",
            "movie IDs",
            "recent financial data",
            "TMDB metadata"
        ]
    }
}


print("=" * 80)
print("SOURCE PRIORITY")
print("=" * 80)

for category, order in source_priority.items():
    print(
        f"{category:20}: "
        f"{' > '.join(order)}"
    )


print("\n" + "=" * 80)
print("SOURCE ROLES")
print("=" * 80)

for source, details in source_roles.items():

    print(
        f"\nDataset {source}: "
        f"{details['name']}"
    )

    print(
        f"Role: {details['role']}"
    )

    print(
        "Strengths:",
        ", ".join(details["strengths"])
    )

SOURCE PRIORITY
financial           : A > B > C
modern_metadata     : C > B > A
tmdb_metadata       : C > B
historical_metadata : A > B > C

SOURCE ROLES

Dataset A: Cleaned Industry Dataset
Role: Historical financial backbone
Strengths: budget, gross revenue, historical coverage, clean financial observations

Dataset B: TMDB Relational Dataset
Role: Relational metadata and enrichment
Strengths: TMDB metadata, genre relationships, cast and crew relationships, production metadata

Dataset C: TMDB Recent Dataset
Role: Modern movie coverage and enrichment
Strengths: modern movie coverage, movie IDs, recent financial data, TMDB metadata


## 06_CANONICAL_COLUMN_SCHEMA

In [33]:
canonical_columns = [
    "movie_key",
    "tmdb_id",
    "title",
    "original_title",
    "release_date",
    "release_year",

    "budget",
    "revenue",

    "genre",
    "director",
    "writer",
    "star",
    "country",

    "runtime",
    "rating",

    "production_company",

    "popularity",
    "vote_average",
    "vote_count",

    "source_a",
    "source_b",
    "source_c"
]


print("=" * 80)
print("CANONICAL INTEGRATED SCHEMA")
print("=" * 80)

for number, column in enumerate(
    canonical_columns,
    start=1
):
    print(
        f"{number:02}. {column}"
    )

CANONICAL INTEGRATED SCHEMA
01. movie_key
02. tmdb_id
03. title
04. original_title
05. release_date
06. release_year
07. budget
08. revenue
09. genre
10. director
11. writer
12. star
13. country
14. runtime
15. rating
16. production_company
17. popularity
18. vote_average
19. vote_count
20. source_a
21. source_b
22. source_c


## 07_DEFINING_SOURCE_TO_CANONICAL_MAPPING

In [34]:
column_mapping = {

    "A": {
        "name": "title",
        "released": "release_date",
        "year": "release_year",
        "budget": "budget",
        "gross": "revenue",
        "genre": "genre",
        "director": "director",
        "writer": "writer",
        "star": "star",
        "country": "country",
        "runtime": "runtime",
        "rating": "rating"
    },

    "B": {
        "title": "title",
        "original_title": "original_title",
        "release_date": "release_date",
        "budget": "budget",
        "revenue": "revenue"
    },

    "C": {
        "movie_id": "tmdb_id",
        "title": "title",
        "release_date": "release_date",
        "budget": "budget",
        "revenue": "revenue"
    }
}

## 08_VALIDATE_COLUMN_MAPPING

In [30]:
source_dataframes = {
    "A": cleaned_movies,
    "B": tmdb_relational,
    "C": tmdb_recent
}


print("=" * 80)
print("COLUMN MAPPING VALIDATION")
print("=" * 80)


mapping_valid = True


for source, mapping in column_mapping.items():

    df = source_dataframes[source]

    print(
        f"\nDataset {source}"
    )

    print("-" * 40)

    for source_column, canonical_column in mapping.items():

        exists = (
            source_column
            in df.columns
        )

        status = (
            "PASS"
            if exists
            else "MISSING"
        )

        print(
            f"{source_column:25} "
            f"-> {canonical_column:20} "
            f"{status}"
        )

        if not exists:
            mapping_valid = False


print("\n" + "=" * 80)

print(
    "OVERALL STATUS:",
    "PASS"
    if mapping_valid
    else "REVIEW REQUIRED"
)

COLUMN MAPPING VALIDATION

Dataset A
----------------------------------------
name                      -> title                PASS
released                  -> release_date         PASS
year                      -> release_year         PASS
budget                    -> budget               PASS
gross                     -> revenue              PASS
genre                     -> genre                PASS
director                  -> director             PASS
writer                    -> writer               PASS
star                      -> star                 PASS
country                   -> country              PASS
runtime                   -> runtime              PASS
rating                    -> rating               PASS

Dataset B
----------------------------------------
title                     -> title                PASS
original_title            -> original_title       PASS
release_date              -> release_date         PASS
budget                    -> budget          

## 09_STANDARDIZE_TITLES

In [29]:
def normalize_movie_title(title):

    if pd.isna(title):
        return pd.NA

    title = str(
        title
    ).strip().lower()

    title = unicodedata.normalize(
        "NFKD",
        title
    )

    title = "".join(
        character
        for character in title
        if not unicodedata.combining(
            character
        )
    )

    title = re.sub(
        r"[^a-z0-9\s]",
        " ",
        title
    )

    title = re.sub(
        r"\s+",
        " ",
        title
    ).strip()

    if title == "":
        return pd.NA

    return title

## EXTRACT RELEASE YEAR

In [28]:
def get_release_year(
    df,
    year_column=None,
    release_date_column=None
):

    if (
        year_column is not None
        and year_column in df.columns
    ):

        return pd.to_numeric(
            df[year_column],
            errors="coerce"
        ).astype("Int64")


    if (
        release_date_column is not None
        and release_date_column in df.columns
    ):

        return pd.to_datetime(
            df[release_date_column],
            errors="coerce"
        ).dt.year.astype("Int64")


    return pd.Series(
        pd.NA,
        index=df.index,
        dtype="Int64"
    )

## 11_STANDARDIZE_DATASET_A

In [27]:
a_standard = pd.DataFrame(
    index=cleaned_movies.index
)


a_standard["title"] = (
    cleaned_movies["name"]
)

a_standard["release_date"] = (
    cleaned_movies["released"]
)

a_standard["release_year"] = (
    get_release_year(
        cleaned_movies,
        year_column="year",
        release_date_column="released"
    )
)

a_standard["budget"] = pd.to_numeric(
    cleaned_movies["budget"],
    errors="coerce"
)

a_standard["revenue"] = pd.to_numeric(
    cleaned_movies["gross"],
    errors="coerce"
)


for column in [
    "genre",
    "director",
    "writer",
    "star",
    "country",
    "runtime",
    "rating"
]:

    if column in cleaned_movies.columns:

        a_standard[column] = (
            cleaned_movies[column]
        )


a_standard["_normalized_title"] = (
    a_standard["title"]
    .apply(normalize_movie_title)
)


a_standard["movie_key"] = (
    a_standard["_normalized_title"].astype("string")
    + " | "
    + a_standard["release_year"].astype("string")
)


a_standard["source_a"] = True
a_standard["source_b"] = False
a_standard["source_c"] = False


print(
    "Dataset A standardized:",
    a_standard.shape
)

display(
    a_standard.head()
)

Dataset A standardized: (7668, 17)


,title,release_date,release_year,budget,revenue,genre,director,writer,star,country,runtime,rating,_normalized_title,movie_key,source_a,source_b,source_c
0,The Shining,1980-06-13,1980,19000000.0,46998772.0,Drama,Stanley Kubrick,Stephen King,Jack Nicholson,United Kingdom,146.0,R,the shining,the shining | 1980,True,False,False
1,The Blue Lagoon,1980-07-02,1980,4500000.0,58853106.0,Adventure,Randal Kleiser,Henry De Vere Stacpoole,Brooke Shields,United States,104.0,R,the blue lagoon,the blue lagoon | 1980,True,False,False
2,Star Wars: Episode V - The Empire Strikes Back,1980-06-20,1980,18000000.0,538375067.0,Action,Irvin Kershner,Leigh Brackett,Mark Hamill,United States,124.0,PG,star wars episode v the empire strikes back,star wars episode v the empire strikes back | ...,True,False,False
3,Airplane!,1980-07-02,1980,3500000.0,83453539.0,Comedy,Jim Abrahams,Jim Abrahams,Robert Hays,United States,88.0,PG,airplane,airplane | 1980,True,False,False
4,Caddyshack,1980-07-25,1980,6000000.0,39846344.0,Comedy,Harold Ramis,Brian Doyle-Murray,Chevy Chase,United States,98.0,R,caddyshack,caddyshack | 1980,True,False,False


## 12_STANDARDIZE_DATASET_B

In [26]:
b_standard = pd.DataFrame(
    index=tmdb_relational.index
)


b_standard["title"] = (
    tmdb_relational["title"]
)


if "original_title" in tmdb_relational.columns:

    b_standard["original_title"] = (
        tmdb_relational[
            "original_title"
        ]
    )


b_standard["release_date"] = (
    tmdb_relational[
        "release_date"
    ]
)


b_standard["release_year"] = (
    get_release_year(
        tmdb_relational,
        release_date_column="release_date"
    )
)


b_standard["budget"] = pd.to_numeric(
    tmdb_relational["budget"],
    errors="coerce"
)


b_standard["revenue"] = pd.to_numeric(
    tmdb_relational["revenue"],
    errors="coerce"
)


b_standard["_normalized_title"] = (
    b_standard["title"]
    .apply(normalize_movie_title)
)


b_standard["movie_key"] = (
    b_standard["_normalized_title"].astype("string")
    + " | "
    + b_standard["release_year"].astype("string")
)


invalid_key = (
    b_standard["_normalized_title"].isna()
    | b_standard["release_year"].isna()
)

b_standard.loc[
    invalid_key,
    "movie_key"
] = pd.NA


b_standard["source_a"] = False
b_standard["source_b"] = True
b_standard["source_c"] = False


print(
    "Dataset B standardized:",
    b_standard.shape
)

display(
    b_standard.head()
)

Dataset B standardized: (9771, 11)


,title,original_title,release_date,release_year,budget,revenue,_normalized_title,movie_key,source_a,source_b,source_c
0,Ariel,Ariel,1988-10-21,1988,0.0,0.0,ariel,ariel | 1988,False,True,False
1,Star Wars,Star Wars,1977-05-25,1977,11000000.0,775398007.0,star wars,star wars | 1977,False,True,False
2,Finding Nemo,Finding Nemo,2003-05-30,2003,94000000.0,940335536.0,finding nemo,finding nemo | 2003,False,True,False
3,Forrest Gump,Forrest Gump,1994-06-23,1994,55000000.0,677387716.0,forrest gump,forrest gump | 1994,False,True,False
4,American Beauty,American Beauty,1999-09-15,1999,15000000.0,356296601.0,american beauty,american beauty | 1999,False,True,False


## 13_STANDARIZE_DATASET_C

In [25]:
c_standard = pd.DataFrame(
    index=tmdb_recent.index
)


c_standard["tmdb_id"] = (
    tmdb_recent[
        "movie_id"
    ]
)


c_standard["title"] = (
    tmdb_recent[
        "title"
    ]
)


if "original_title" in tmdb_recent.columns:

    c_standard["original_title"] = (
        tmdb_recent[
            "original_title"
        ]
    )


c_standard["release_date"] = (
    tmdb_recent[
        "release_date"
    ]
)


c_standard["release_year"] = (
    get_release_year(
        tmdb_recent,
        release_date_column="release_date"
    )
)


c_standard["budget"] = pd.to_numeric(
    tmdb_recent["budget"],
    errors="coerce"
)


c_standard["revenue"] = pd.to_numeric(
    tmdb_recent["revenue"],
    errors="coerce"
)


c_standard["_normalized_title"] = (
    c_standard["title"]
    .apply(normalize_movie_title)
)


c_standard["movie_key"] = (
    c_standard["_normalized_title"].astype("string")
    + " | "
    + c_standard["release_year"].astype("string")
)


c_standard["source_a"] = False
c_standard["source_b"] = False
c_standard["source_c"] = True


print(
    "Dataset C standardized:",
    c_standard.shape
)

display(
    c_standard.head()
)

Dataset C standardized: (17978, 11)


,tmdb_id,title,release_date,release_year,budget,revenue,_normalized_title,movie_key,source_a,source_b,source_c
0,113727,Dark Seduction,2010-01-01,2010,0,0,dark seduction,dark seduction | 2010,False,False,True
1,67250,12,2010-01-01,2010,0,0,12,12 | 2010,False,False,True
2,43615,"Lula, the Son of Brazil",2010-01-01,2010,9500000,3785593,lula the son of brazil,lula the son of brazil | 2010,False,False,True
3,38883,Protect and Serve,2010-01-03,2010,0,3807808,protect and serve,protect and serve | 2010,False,False,True
4,62105,Re-Cut,2010-01-04,2010,0,0,re cut,re cut | 2010,False,False,True


## 14_VALIDATE_STANDARDIZED_DATASETS

In [23]:
standardized_sources = {
    "A": a_standard,
    "B": b_standard,
    "C": c_standard
}


print("=" * 80)
print("STANDARDIZED DATASET VALIDATION")
print("=" * 80)


for source, df in standardized_sources.items():

    print(
        f"\nDataset {source}"
    )

    print("-" * 40)

    print(
        f"Rows:             "
        f"{len(df):,}"
    )

    print(
        f"Valid movie keys: "
        f"{df['movie_key'].notna().sum():,}"
    )

    print(
        f"Unique keys:      "
        f"{df['movie_key'].nunique():,}"
    )

    print(
        f"Duplicate keys:   "
        f"{df[df['movie_key'].duplicated(keep=False)]['movie_key'].nunique():,}"
    )

STANDARDIZED DATASET VALIDATION

Dataset A
----------------------------------------
Rows:             7,668
Valid movie keys: 7,668
Unique keys:      7,668
Duplicate keys:   0

Dataset B
----------------------------------------
Rows:             9,771
Valid movie keys: 9,708
Unique keys:      9,693
Duplicate keys:   15

Dataset C
----------------------------------------
Rows:             17,978
Valid movie keys: 17,978
Unique keys:      17,931
Duplicate keys:   45


## 15_INVESTIGATE_DATASET_B_INVALID_KEYS

In [35]:
b_invalid_keys = b_standard[
    b_standard["movie_key"].isna()
].copy()


print("=" * 80)
print("DATASET B - INVALID MATCH KEYS")
print("=" * 80)

print(
    f"Invalid rows: "
    f"{len(b_invalid_keys):,}"
)


print("\nREASONS")
print("-" * 40)

missing_title = (
    b_invalid_keys[
        "_normalized_title"
    ].isna().sum()
)

missing_year = (
    b_invalid_keys[
        "release_year"
    ].isna().sum()
)


print(
    f"Missing/blank title: "
    f"{missing_title:,}"
)

print(
    f"Missing release year: "
    f"{missing_year:,}"
)


columns_to_show = [
    column
    for column in [
        "title",
        "original_title",
        "release_date",
        "release_year",
        "_normalized_title",
        "movie_key"
    ]
    if column in b_invalid_keys.columns
]


display(
    b_invalid_keys[
        columns_to_show
    ]
)

DATASET B - INVALID MATCH KEYS
Invalid rows: 63

REASONS
----------------------------------------
Missing/blank title: 1
Missing release year: 62


,title,original_title,release_date,release_year,_normalized_title,movie_key
5236,Highlander,Highlander,NaN,<NA>,highlander,<NA>
5883,Painkiller Jane,Painkiller Jane,NaN,<NA>,painkiller jane,<NA>
6100,Mama 2,Mama 2,NaN,<NA>,mama 2,<NA>
6130,XXXXXXX,XXXXXXX,None,<NA>,xxxxxxx,<NA>
6131,2016-09-01,13,0,<NA>,2016 09 01,<NA>
...,...,...,...,...,...,...
9681,Spermateket,Spermateket,NaN,<NA>,spermateket,<NA>
9703,Liked,Liked,NaN,<NA>,liked,<NA>
9710,Sanctuary,Deca bogova,NaN,<NA>,sanctuary,<NA>
9728,Audition,Audition,NaN,<NA>,audition,<NA>


## 16_CLASSIFY_MATCH_KEY_QUALITY

In [36]:
def classify_match_keys(df):
    # Classify each row's matching key as:
    # - missing
    # - unique
    # - ambiguous

    result = df.copy()

    key_counts = (
        result[
            "movie_key"
        ]
        .value_counts(
            dropna=True
        )
    )

    result["match_key_count"] = (
        result["movie_key"]
        .map(key_counts)
        .astype("Int64")
    )

    result["match_status"] = "unique"

    result.loc[
        result["movie_key"].isna(),
        "match_status"
    ] = "missing"

    result.loc[
        result["match_key_count"].gt(1),
        "match_status"
    ] = "ambiguous"

    return result


a_standard = classify_match_keys(
    a_standard
)

b_standard = classify_match_keys(
    b_standard
)

c_standard = classify_match_keys(
    c_standard
)

## 17_MATCH_KEY_STATUS_SUMMARY

In [37]:
print("=" * 80)
print("MATCH-KEY STATUS SUMMARY")
print("=" * 80)


for source_name, df in {
    "Dataset A": a_standard,
    "Dataset B": b_standard,
    "Dataset C": c_standard
}.items():

    print(
        f"\n{source_name}"
    )

    print("-" * 40)

    status_counts = (
        df["match_status"]
        .value_counts()
    )

    for status, count in status_counts.items():

        print(
            f"{status:12}: "
            f"{count:,}"
        )

MATCH-KEY STATUS SUMMARY

Dataset A
----------------------------------------
unique      : 7,668

Dataset B
----------------------------------------
unique      : 9,678
missing     : 63
ambiguous   : 30

Dataset C
----------------------------------------
unique      : 17,886
ambiguous   : 92


## 18_INSPECT_AMBIGUOUS_DATSET_B_RECORDS

In [38]:
b_ambiguous = b_standard[
    b_standard[
        "match_status"
    ].eq("ambiguous")
].copy()


print("=" * 80)
print("DATASET B - AMBIGUOUS MATCH RECORDS")
print("=" * 80)

print(
    f"Rows: "
    f"{len(b_ambiguous):,}"
)

print(
    f"Keys: "
    f"{b_ambiguous['movie_key'].nunique():,}"
)


display_columns = [
    column
    for column in [
        "movie_key",
        "title",
        "original_title",
        "release_date",
        "release_year",
        "budget",
        "revenue"
    ]
    if column in b_ambiguous.columns
]


display(
    b_ambiguous[
        display_columns
    ].sort_values(
        "movie_key"
    )
)

DATASET B - AMBIGUOUS MATCH RECORDS
Rows: 30
Keys: 15


,movie_key,title,original_title,release_date,release_year,budget,revenue
1156,animal | 2005,Animal,Animal,2005-05-01,2005,0.0,0.0
3502,animal | 2005,Animal,Animal,2005-01-11,2005,0.0,0.0
9102,beauty | 2018,Beauty,Beauty,2018-05-09,2018,0.0,0.0
7417,beauty | 2018,Beauty,Beauty,2018-01-07,2018,0.0,0.0
8816,companion | 2025,Companion,Sahela,2025-03-20,2025,0.0,0.0
8552,companion | 2025,Companion,Companion,2025-01-22,2025,10000000.0,36869122.0
5177,consumed | 2015,Consumed,Consumed,2015-06-01,2015,0.0,0.0
5483,consumed | 2015,Consumed,Consumed,2015-07-09,2015,50000.0,0.0
1943,darling | 2007,Darling,Darling,2007-02-09,2007,0.0,0.0
2890,darling | 2007,Darling,Darling,2007-11-07,2007,0.0,0.0


## 19_INSPECT_AMBIGUOUS_DATASET_C_RECORDS

In [39]:
c_ambiguous = c_standard[
    c_standard[
        "match_status"
    ].eq("ambiguous")
].copy()


print("=" * 80)
print("DATASET C - AMBIGUOUS MATCH RECORDS")
print("=" * 80)

print(
    f"Rows: "
    f"{len(c_ambiguous):,}"
)

print(
    f"Keys: "
    f"{c_ambiguous['movie_key'].nunique():,}"
)


display_columns = [
    column
    for column in [
        "movie_key",
        "tmdb_id",
        "title",
        "release_date",
        "release_year",
        "budget",
        "revenue"
    ]
    if column in c_ambiguous.columns
]


display(
    c_ambiguous[
        display_columns
    ].sort_values(
        "movie_key"
    )
)

DATASET C - AMBIGUOUS MATCH RECORDS
Rows: 92
Keys: 45


,movie_key,tmdb_id,title,release_date,release_year,budget,revenue
3599,1 | 2013,217316,1,2013-09-30,2013,0,0
3569,1 | 2013,176068,+1,2013-09-20,2013,0,0
1684,11 11 11 | 2011,79078,11/11/11,2011-11-01,2011,0,0
1716,11 11 11 | 2011,51248,11-11-11,2011-11-11,2011,0,6963872
1291,a better life | 2011,55720,A Better Life,2011-06-24,2011,10000000,1800000
...,...,...,...,...,...,...,...
8539,veronica | 2017,441701,Veronica,2017-08-25,2017,0,0
1536,war of the buttons | 2011,74945,War of the Buttons,2011-09-21,2011,0,15000000
1504,war of the buttons | 2011,74944,War of the Buttons,2011-09-14,2011,0,12000000
10171,zoo | 2018,552504,Zoo,2018-10-05,2018,0,0


## 20_CREATE_EXACT_DATE_MATCH_KEYS

In [ ]:
def add_strong_match_key(df):
    #Add an exact title + release-date matching key.

    result = df.copy()

    result["_match_date"] = (
        pd.to_datetime(
            result["release_date"],
            errors="coerce"
        )
        .dt.strftime("%Y-%m-%d")
    )

    valid = (
        result["_normalized_title"].notna()
        & result["_match_date"].notna()
    )

    result["_strong_match_key"] = pd.NA

    result.loc[
        valid,
        "_strong_match_key"
    ] = (
        result.loc[
            valid,
            "_normalized_title"
        ].astype(str)
        + " | "
        + result.loc[
            valid,
            "_match_date"
        ].astype(str)
    )

    return result


a_standard = add_strong_match_key(
    a_standard
)

b_standard = add_strong_match_key(
    b_standard
)

c_standard = add_strong_match_key(
    c_standard
)

## 21_STRONG_KEY_QUALITY_AUDIT

In [41]:
def strong_key_audit(
    df,
    dataset_name
):
    """
    Inspect uniqueness of exact title + release-date keys.
    """

    valid = df[
        df["_strong_match_key"].notna()
    ]

    duplicate_keys = (
        valid[
            valid.duplicated(
                "_strong_match_key",
                keep=False
            )
        ][
            "_strong_match_key"
        ].nunique()
    )

    print(
        f"\n{dataset_name}"
    )

    print("-" * 50)

    print(
        f"Rows:                 "
        f"{len(df):,}"
    )

    print(
        f"Valid strong keys:    "
        f"{len(valid):,}"
    )

    print(
        f"Unique strong keys:   "
        f"{valid['_strong_match_key'].nunique():,}"
    )

    print(
        f"Duplicate strong keys:"
        f" {duplicate_keys:,}"
    )


print("=" * 80)
print("EXACT-DATE MATCH-KEY QUALITY")
print("=" * 80)


strong_key_audit(
    a_standard,
    "Dataset A"
)

strong_key_audit(
    b_standard,
    "Dataset B"
)

strong_key_audit(
    c_standard,
    "Dataset C"
)

EXACT-DATE MATCH-KEY QUALITY

Dataset A
--------------------------------------------------
Rows:                 7,668
Valid strong keys:    7,609
Unique strong keys:   7,609
Duplicate strong keys: 0

Dataset B
--------------------------------------------------
Rows:                 9,771
Valid strong keys:    9,708
Unique strong keys:   9,708
Duplicate strong keys: 0

Dataset C
--------------------------------------------------
Rows:                 17,978
Valid strong keys:    17,978
Unique strong keys:   17,978
Duplicate strong keys: 0


## 22_SAFE_CROSS_SOURCE_MATCHING_FUNCTION

In [42]:
def safe_cross_source_match(
    left_df,
    right_df,
    left_name,
    right_name
):
    # Match two datasets conservatively.

    # Tier 1:
    #   Unique title + year.

    # Tier 2:
    #   Unique title + exact release date
    #   for records not already matched.

    left = (
        left_df
        .copy()
        .reset_index()
        .rename(
            columns={
                "index": f"{left_name}_row"
            }
        )
    )

    right = (
        right_df
        .copy()
        .reset_index()
        .rename(
            columns={
                "index": f"{right_name}_row"
            }
        )
    )


    # TIER 1 — UNIQUE TITLE + YEAR

    left_year = left[
        left["match_status"].eq("unique")
        & left["movie_key"].notna()
    ][
        [
            f"{left_name}_row",
            "movie_key"
        ]
    ]


    right_year = right[
        right["match_status"].eq("unique")
        & right["movie_key"].notna()
    ][
        [
            f"{right_name}_row",
            "movie_key"
        ]
    ]


    year_matches = left_year.merge(
        right_year,
        on="movie_key",
        how="inner"
    )


    year_matches["match_method"] = (
        "title_year"
    )


    # RECORDS ALREADY MATCHED

    matched_left = set(
        year_matches[
            f"{left_name}_row"
        ]
    )

    matched_right = set(
        year_matches[
            f"{right_name}_row"
        ]
    )


    # STRONG-KEY COUNTS

    left_strong_counts = (
        left[
            "_strong_match_key"
        ]
        .value_counts(
            dropna=True
        )
    )


    right_strong_counts = (
        right[
            "_strong_match_key"
        ]
        .value_counts(
            dropna=True
        )
    )


    left["_strong_count"] = (
        left[
            "_strong_match_key"
        ]
        .map(left_strong_counts)
    )


    right["_strong_count"] = (
        right[
            "_strong_match_key"
        ]
        .map(right_strong_counts)
    )


    # TIER 2 — TITLE + EXACT DATE

    left_date = left[
        ~left[
            f"{left_name}_row"
        ].isin(matched_left)
        & left[
            "_strong_match_key"
        ].notna()
        & left[
            "_strong_count"
        ].eq(1)
    ][
        [
            f"{left_name}_row",
            "_strong_match_key"
        ]
    ]


    right_date = right[
        ~right[
            f"{right_name}_row"
        ].isin(matched_right)
        & right[
            "_strong_match_key"
        ].notna()
        & right[
            "_strong_count"
        ].eq(1)
    ][
        [
            f"{right_name}_row",
            "_strong_match_key"
        ]
    ]


    date_matches = left_date.merge(
        right_date,
        on="_strong_match_key",
        how="inner"
    )


    date_matches["match_method"] = (
        "title_exact_date"
    )


    # STANDARDIZE OUTPUT

    year_output = year_matches[
        [
            f"{left_name}_row",
            f"{right_name}_row",
            "match_method"
        ]
    ]


    date_output = date_matches[
        [
            f"{left_name}_row",
            f"{right_name}_row",
            "match_method"
        ]
    ]


    matches = pd.concat(
        [
            year_output,
            date_output
        ],
        ignore_index=True
    )


    return matches

## 23_MATCHING_ALL_3_DATASET_PAIRS

In [43]:
def safe_cross_source_match(
    left_df,
    right_df,
    left_name,
    right_name
):
    """
    Match two datasets conservatively.

    Tier 1:
        Unique title + year.

    Tier 2:
        Unique title + exact release date
        for records not already matched.
    """

    left = (
        left_df
        .copy()
        .reset_index()
        .rename(
            columns={
                "index": f"{left_name}_row"
            }
        )
    )

    right = (
        right_df
        .copy()
        .reset_index()
        .rename(
            columns={
                "index": f"{right_name}_row"
            }
        )
    )


    # --------------------------------------------------------
    # TIER 1 — UNIQUE TITLE + YEAR
    # --------------------------------------------------------

    left_year = left[
        left["match_status"].eq("unique")
        & left["movie_key"].notna()
    ][
        [
            f"{left_name}_row",
            "movie_key"
        ]
    ]


    right_year = right[
        right["match_status"].eq("unique")
        & right["movie_key"].notna()
    ][
        [
            f"{right_name}_row",
            "movie_key"
        ]
    ]


    year_matches = left_year.merge(
        right_year,
        on="movie_key",
        how="inner"
    )


    year_matches["match_method"] = (
        "title_year"
    )


    # --------------------------------------------------------
    # RECORDS ALREADY MATCHED
    # --------------------------------------------------------

    matched_left = set(
        year_matches[
            f"{left_name}_row"
        ]
    )

    matched_right = set(
        year_matches[
            f"{right_name}_row"
        ]
    )


    # --------------------------------------------------------
    # STRONG-KEY COUNTS
    # --------------------------------------------------------

    left_strong_counts = (
        left[
            "_strong_match_key"
        ]
        .value_counts(
            dropna=True
        )
    )


    right_strong_counts = (
        right[
            "_strong_match_key"
        ]
        .value_counts(
            dropna=True
        )
    )


    left["_strong_count"] = (
        left[
            "_strong_match_key"
        ]
        .map(left_strong_counts)
    )


    right["_strong_count"] = (
        right[
            "_strong_match_key"
        ]
        .map(right_strong_counts)
    )


    # --------------------------------------------------------
    # TIER 2 — TITLE + EXACT DATE
    # --------------------------------------------------------

    left_date = left[
        ~left[
            f"{left_name}_row"
        ].isin(matched_left)
        & left[
            "_strong_match_key"
        ].notna()
        & left[
            "_strong_count"
        ].eq(1)
    ][
        [
            f"{left_name}_row",
            "_strong_match_key"
        ]
    ]


    right_date = right[
        ~right[
            f"{right_name}_row"
        ].isin(matched_right)
        & right[
            "_strong_match_key"
        ].notna()
        & right[
            "_strong_count"
        ].eq(1)
    ][
        [
            f"{right_name}_row",
            "_strong_match_key"
        ]
    ]


    date_matches = left_date.merge(
        right_date,
        on="_strong_match_key",
        how="inner"
    )


    date_matches["match_method"] = (
        "title_exact_date"
    )


    # --------------------------------------------------------
    # STANDARDIZE OUTPUT
    # --------------------------------------------------------

    year_output = year_matches[
        [
            f"{left_name}_row",
            f"{right_name}_row",
            "match_method"
        ]
    ]


    date_output = date_matches[
        [
            f"{left_name}_row",
            f"{right_name}_row",
            "match_method"
        ]
    ]


    matches = pd.concat(
        [
            year_output,
            date_output
        ],
        ignore_index=True
    )


    return matches